# Ce que fait la SVD

| Module | Durée | Prérequis |
|---|---|---|
| M8 – Décomposition en valeurs singulières et données | 75 min | M6 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une graphiste applique l'effet « Incliner » de son logiciel (le cisaillement du module M3) à un badge rond. Le badge devient une **ellipse** penchée. Pour commander l'autocollant, l'imprimeur lui demande la **longueur des deux axes** de l'ellipse et leur **orientation**.

Au module M7, on aurait cherché les directions qui ne tournent pas. Mais le cisaillement n'en a qu'une, l'horizontale, et ce n'est pas l'axe de l'ellipse. Les vecteurs propres ne suffisent donc pas à décrire ce que fait cette matrice.

$$
A = \begin{pmatrix} 1 & 0{,}8 \\ 0 & 1 \end{pmatrix}
$$

**Les questions : comment trouver les axes de l'ellipse, et de quelles directions du badge ils proviennent ? Existe-t-il une description qui marche pour toute matrice, même rectangulaire ?**

**Activité** : Le cisaillement conserve les aires (module M4 : $\det A = 1$). Que pouvez-vous en déduire sur le produit des deux demi-axes de l'ellipse, si le badge est un cercle de rayon 1 ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_demi_grand_axe = 1.3        # longueur du demi-grand axe de l'ellipse (modifiez)
pari_angle_grand_axe = 20        # angle du grand axe avec l'horizontale, en degrés (modifiez)

## ③ Abstraire : la décomposition en valeurs singulières

Toute matrice $A$, de n'importe quelle taille, s'écrit

$$
A = U\,\Sigma\,V^{\mathsf{T}} ,
$$

- $V$ et $U$ ont des colonnes **orthonormées** (de longueur 1, deux à deux perpendiculaires) : ce sont des rotations, éventuellement suivies d'un miroir ;
- $\Sigma$ est diagonale, avec des nombres positifs rangés par ordre décroissant, $\sigma_1 \geq \sigma_2 \geq \dots \geq 0$ : les **valeurs singulières**.

Géométriquement : tourner, étirer selon des axes perpendiculaires, tourner.

![Schéma : la SVD décompose A en trois étapes : une rotation V transposée amène v1 et v2 sur les axes, Sigma étire selon les axes par sigma1 et sigma2, U tourne le résultat ; le cercle unité devient l'ellipse de A](../../assets/schemas/schema-svd-geometrie.svg)

Colonne par colonne, cela s'écrit

$$
A\,\mathbf{v}_i = \sigma_i\,\mathbf{u}_i :
$$

la direction $\mathbf{v}_i$ du départ est envoyée sur l'axe $\mathbf{u}_i$ de l'ellipse, avec la longueur $\sigma_i$. Contrairement aux vecteurs propres, les $\mathbf{v}_i$ sont toujours perpendiculaires entre eux, les $\mathbf{u}_i$ aussi, et les $\sigma_i$ sont toujours des nombres réels positifs.

**Comment les calculer à la main ?** Comme $A^{\mathsf{T}}A = V\Sigma^2V^{\mathsf{T}}$, les $\sigma_i^2$ sont les valeurs propres de la matrice symétrique $A^{\mathsf{T}}A$, et les $\mathbf{v}_i$ ses vecteurs propres (module M7).

**Activité** : Pour $A = \begin{pmatrix} 3 & 0 \\ 4 & 5 \end{pmatrix}$, calculez $A^{\mathsf{T}}A$, puis ses valeurs propres. Vous devez trouver $\sigma_1 = \sqrt{45}$ et $\sigma_2 = \sqrt{5}$. Vérifiez que $\sigma_1\sigma_2 = \lvert \det A \rvert$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### La SVD du cisaillement

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[1, 0.8], [0, 1]])
U, s, Vt = np.linalg.svd(A)          # attention : NumPy renvoie V transposée, et s sous forme de vecteur
print("valeurs singulières :", s.round(4))
print("U =\n", U.round(4))
print("V^T =\n", Vt.round(4))
print("U Σ V^T = A ?", np.allclose(U @ np.diag(s) @ Vt, A))
print("colonnes de U orthonormées ?", np.allclose(U.T @ U, np.eye(2)), "  de V ?", np.allclose(Vt @ Vt.T, np.eye(2)))

V = Vt.T
for i in range(2):
    print(f"A v{i + 1} = σ{i + 1} u{i + 1} ? {np.allclose(A @ V[:, i], s[i] * U[:, i])}   "
          f"axe de l'ellipse à {np.degrees(np.arctan2(U[1, i], U[0, i])) % 180:.1f}°")

### Les trois étapes, en images

In [ ]:
t = np.linspace(0, 2 * np.pi, 200)
cercle = np.vstack([np.cos(t), np.sin(t)])
etapes = [("départ", np.eye(2)), ("1. tourner : Vᵀ", Vt), ("2. étirer : Σ Vᵀ", np.diag(s) @ Vt), ("3. tourner : U Σ Vᵀ = A", A)]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (titre, M) in zip(axes, etapes):
    ax.plot(*(M @ cercle), color=LIOUAI["navy"])
    for i, couleur in zip(range(2), [LIOUAI["vert"], LIOUAI["jaune"]]):
        image = M @ V[:, i]
        ax.annotate("", image, (0, 0), arrowprops=dict(arrowstyle="-|>", color=couleur, lw=2))
    ax.set_xlim(-1.7, 1.7); ax.set_ylim(-1.7, 1.7); ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title(titre)
plt.tight_layout(); plt.show()

### Et les vecteurs propres ?

In [ ]:
valeurs, vecteurs = np.linalg.eig(A)
print("valeurs propres :", np.real_if_close(valeurs))
print("vecteurs propres (en colonnes) :\n", np.real_if_close(vecteurs).round(6))
print("\nAngle entre les deux vecteurs propres trouvés :",
      f"{np.degrees(np.arccos(abs(np.real(vecteurs[:, 0] @ vecteurs[:, 1])))):.2e}°")

**Activité** : Pourquoi `eig` renvoie-t-il deux fois (presque) le même vecteur ? Pour quelles matrices les vecteurs propres et les vecteurs singuliers coïncident-ils ? Testez avec la matrice symétrique du tissu du notebook 7.1.

Indiquez vos réponses et justifiez-les.

### Des matrices rectangulaires

La SVD s'applique aussi aux matrices qui changent de dimension. Une matrice $3 \times 2$ envoie le plan dans l'espace ; une matrice $2 \times 3$ projette l'espace sur un plan, comme le moteur 3D du module M3.

In [ ]:
plonger = np.array([[1, 0], [0, 1], [0.5, 0.5]])         # du plan vers l'espace
projeter = np.array([[1, 0, 0.3], [0, 1, 0.3]])           # de l'espace vers le plan
for nom, M in [("3 x 2 (plonger)", plonger), ("2 x 3 (projeter)", projeter)]:
    Um, sm, Vtm = np.linalg.svd(M)
    print(f"{nom} : U {Um.shape}, valeurs singulières {sm.round(4)}, V^T {Vtm.shape}")
print("\nDirection de l'espace écrasée par la projection (dernière ligne de V^T) :", np.linalg.svd(projeter)[2][-1].round(4))
print("Son image :", (projeter @ np.linalg.svd(projeter)[2][-1]).round(12))

### Ce que les valeurs singulières résument

In [ ]:
t30 = np.radians(30)
galerie = {
    "rotation de 30°": np.array([[np.cos(t30), -np.sin(t30)], [np.sin(t30), np.cos(t30)]]),
    "cisaillement": A,
    "étirement × 2 en x": np.array([[2, 0], [0, 1]]),
    "projection sur x": np.array([[1, 0], [0, 0]]),
    "presque singulière": np.array([[1, 1], [1, 1.001]]),
}
print(f"{'transformation':20s} {'σ1':>8s} {'σ2':>9s} {'σ1 σ2':>8s} {'|det|':>8s} {'σ1/σ2':>10s} {'cond':>10s}")
for nom, M in galerie.items():
    sm = np.linalg.svd(M, compute_uv=False)
    rapport = sm[0] / sm[1] if sm[1] > 1e-12 else np.inf
    print(f"{nom:20s} {sm[0]:8.4f} {sm[1]:9.6f} {sm[0] * sm[1]:8.4f} {abs(np.linalg.det(M)):8.4f} "
          f"{rapport:10.1f} {np.linalg.cond(M):10.1f}")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : demi-grand axe {pari_demi_grand_axe}, à {pari_angle_grand_axe}°")
print(f"Résultat   : demi-grand axe {s[0]:.3f}, à {np.degrees(np.arctan2(U[1, 0], U[0, 0])) % 180:.1f}° ; "
      f"demi-petit axe {s[1]:.3f}")

**Activité** :

1. Répondez à l'imprimeur : dimensions de l'ellipse (pour un badge de 5 cm de diamètre), orientation du grand axe.
2. Que représentent les colonnes de $V$ pour la graphiste ? Où sont-elles sur le badge d'origine ?
3. D'après le tableau, comment lit-on sur les valeurs singulières qu'une matrice conserve les longueurs ? qu'elle écrase une direction ? qu'elle est presque singulière ? Faites le lien avec le conditionnement du module M4.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Unicité.** Changez le signe de $\mathbf{u}_1$ et de $\mathbf{v}_1$ en même temps : obtient-on encore une SVD de $A$ ? Que faut-il en conclure quand deux logiciels donnent des vecteurs singuliers de signes différents ?
- **Vérifier autrement.** Les axes de l'ellipse sont les points du contour le plus loin et le plus près de l'origine. Retrouvez $\sigma_1$ et $\sigma_2$ en cherchant le maximum et le minimum de $\lVert A\mathbf{x} \rVert$ sur 10 000 points du cercle.
- **Généraliser.** Écrivez une fonction `svd_par_valeurs_propres(A)` qui calcule la SVD d'une matrice carrée inversible à partir des éléments propres de $A^{\mathsf{T}}A$ (avec `np.linalg.eigh`, prévue pour les matrices symétriques), puis $\mathbf{u}_i = A\mathbf{v}_i / \sigma_i$. Comparez avec `np.linalg.svd` sur la galerie. Pourquoi les logiciels n'utilisent-ils pas cette méthode ? Pensez aux équations normales du module M6.

In [ ]:
def svd_par_valeurs_propres(A):
    # À compléter : renvoie (U, s, Vt)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def svd_par_valeurs_propres(A):
    valeurs, V = np.linalg.eigh(A.T @ A)          # valeurs propres croissantes
    ordre = np.argsort(valeurs)[::-1]
    s = np.sqrt(np.clip(valeurs[ordre], 0, None))
    V = V[:, ordre]
    with np.errstate(divide="ignore", invalid="ignore"):
        U = (A @ V) / s                            # divise chaque colonne par sa valeur singulière
    return U, s, V.T

for nom, M in galerie.items():
    if nom == "projection sur x":
        continue                                   # sigma2 = 0 : la division est impossible
    U2, s2, Vt2 = svd_par_valeurs_propres(M)
    erreur = np.abs(s2 - np.linalg.svd(M, compute_uv=False)).max()
    print(f"{nom:20s} écart sur les σ : {erreur:.1e}   reconstruction : {np.allclose(U2 @ np.diag(s2) @ Vt2, M)}")

contour = A @ cercle
longueurs = np.linalg.norm(A @ np.vstack([np.cos(np.linspace(0, 2 * np.pi, 10_000)),
                                          np.sin(np.linspace(0, 2 * np.pi, 10_000))]), axis=0)
print(f"\nMax et min de |A x| sur le cercle : {longueurs.max():.4f} et {longueurs.min():.4f} ; σ = {s.round(4)}")

eps = 1e-9
B = np.array([[1, 1], [0, eps]])
print(f"\nσ2 exact ≈ {eps / np.sqrt(2):.2e} ; par A^T A : {svd_par_valeurs_propres(B)[1][1]:.2e} ; "
      f"par np.linalg.svd : {np.linalg.svd(B, compute_uv=False)[1]:.2e}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Décrire le concept": 1,
    "Expliquer visuellement": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
    "Corroborer ses résultats": 1,
    "Estimer la solution avant de calculer": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "8.1-ce-que-fait-la-svd", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")